# 03 — Scaffold a New Agent

This notebook walks you through creating a new agent using the scaffolding script,
verifying the generated files, running the generated tests, deploying it locally,
and optionally deleting it.

## Prerequisites

- Python 3.11+
- Dependencies installed (`uv sync --group dev`)

## Configuration

Set the name for your new agent below. Must be kebab-case (e.g., `my-agent`).

In [ ]:
import sys, pathlib

# Ensure the local project root is first on sys.path so our `agents`
# package takes priority over the openai-agents site-package.
_project_root = str(pathlib.Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# Change this to your desired agent name (kebab-case)
AGENT_NAME = "probation-poc-agent"
MODEL = "gpt-4o"

## Step 1: Scaffold the Agent

Run the scaffolding script with `--name` to generate all agent files, test stubs, and registry entry.

In [ ]:
!python ../scripts/create_agent.py --name {AGENT_NAME} --model {MODEL}

## Step 2: Verify Generated Files

Check that all expected files were created in the agent and test directories.

In [ ]:
import os

module_name = AGENT_NAME.replace("-", "_")
agent_dir = f"../agents/{module_name}"
test_dir = f"../tests/{module_name}"

print(f"Agent directory: agents/{module_name}/")
for root, dirs, files in os.walk(agent_dir):
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        print(f"{sub_indent}{f}")

print(f"\nTest directory: tests/{module_name}/")
for root, dirs, files in os.walk(test_dir):
    level = root.replace(test_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        print(f"{sub_indent}{f}")

## Step 3: Inspect Generated Config

View the generated configuration to confirm the agent name, model, and paths.

In [ ]:
config_path = f"../agents/{module_name}/config.py"
print(open(config_path).read())

## Step 4: Inspect Generated Instructions

View the placeholder system prompt. You'll customise this with your agent's role and capabilities.

In [ ]:
instructions_path = f"../agents/{module_name}/instructions.md"
print(open(instructions_path).read())

## Step 5: Run Generated Tests

Run the unit tests generated for your new agent to verify the scaffolding is correct.

In [ ]:
!python -m pytest ../tests/{module_name}/ -v -m "not integration"

## Step 6: Run Your New Agent

Assemble and run the agent locally to verify it works end-to-end.

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

# Reload registry to pick up the new agent
import importlib
import agents.registry
importlib.reload(agents.registry)
from agents.registry import REGISTRY

from agents._base.agent_factory import create_agent

entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()
agent = create_agent(config)

print(f"✓ Agent '{config.agent_name}' assembled")
print(f"  Model: {config.agent_deployment_name}")

In [ ]:
# Send a test prompt
result = await agent.run("Hello! Please greet Alice using your greeting tool.")
response_text = result.text if hasattr(result, "text") else str(result)
print(f"Agent response: {response_text}")

## Step 7: Test as HTTP Service (Optional)

You can also test the agent as an HTTP service. Run this in a terminal:

```bash
AGENT_NAME=my-demo-agent python app.py
```

Then test with curl:

```bash
curl -X POST http://localhost:8088/responses \
  -H "Content-Type: application/json" \
  -d '{"input": "Hello!"}'
```

## Step 8: Delete the Agent (Optional)

Remove the scaffolded agent from the codebase — deletes the agent directory,
test directory, and registry entry.

⚠️ **This is destructive and cannot be undone.** Only run this if you want to completely remove the agent.

In [ ]:
# Uncomment to delete the scaffolded agent (this is irreversible)
#!python ../scripts/delete_agent.py --name {AGENT_NAME} --force

## Next Steps

1. Edit `agents/<your_agent>/instructions.md` with your agent's system prompt
2. Add custom tools in `agents/<your_agent>/tools/` — see the [Custom Tools Guide](../docs/custom-tools-guide.md)
3. (Optional) Set defaults in `agents/<your_agent>/config.py` (model, temperature, etc.)
4. See the [Scaffolding Guide](../docs/scaffolding-guide.md) for full reference
5. Continue to **04_deploy_to_aca.ipynb** to deploy your agent to Azure Container Apps
6. See the [Deployment Guide](../docs/deployment-guide.md) for full deployment reference